# Deepo — Machine Translation System
**Seq2Seq LSTM | Multilingual ↔ English (bidirectional)**
> French, Spanish, Arabic, Portuguese ↔ English in one single global model

Sections:
1. Setup & Google Drive
2. Data Extraction
3. Data Cleaning
4. Language Tags
5. CSV Construction
6. Model Training
7. Inference (Translation)

In [ ]:
##!pip install tqdm -q

## 1. Setup & Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE      = os.getcwd()
DATA_DIR  = os.path.join(BASE, 'datasets')
ZIP_DIR   = os.path.join(BASE, 'zip_data')
RAW_DIR   = os.path.join(BASE, 'raw_data')
CLEAN_DIR = os.path.join(BASE, 'clean_data')
CKPT_PATH = '/content/drive/MyDrive/best_multi.pt' 

for d in [ZIP_DIR, RAW_DIR, CLEAN_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('BASE :', BASE)
print('DATA_DIR :', DATA_DIR)
print('CKPT_PATH:', CKPT_PATH)
print('Setup OK')

## 2. Data Extraction
Upload zip files in `/content/zip_data/` (Colab) :
- `fra-eng.zip` → French
- `spa-eng.zip` → Spanish
- `ara-eng.zip` → Arabic
- `por-eng.zip` → Portuguese

then, launch the next case

In [ ]:
import zipfile

# languages to include in the global model, add or remove codes here
KEEP_LANGS = {'fra', 'spa', 'ara', 'por'}

for fname in os.listdir(ZIP_DIR):
    if not fname.endswith('.zip'):
        continue
    lang = fname.replace('-eng.zip', '')
    if lang not in KEEP_LANGS:
        continue  # skip languages not in the list
    with zipfile.ZipFile(os.path.join(ZIP_DIR, fname)) as z:
        for name in z.namelist():
            if name.endswith('.txt') and not name.startswith('_about'):
                z.extract(name, RAW_DIR)
                print(f'Extracted: {name}')

print('Extraction done')

## 3. Data Cleaning
Remove the license column, keep only `source\ttarget`.

In [ ]:
for filename in os.listdir(RAW_DIR):
    if not filename.endswith('.txt'):
        continue

    src_path = os.path.join(RAW_DIR, filename)
    dst_path = os.path.join(CLEAN_DIR, filename)

    with open(src_path, 'r', encoding='utf-8') as src, \
         open(dst_path, 'w', encoding='utf-8') as dst:

        for line in src:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) >= 2:
                dst.write(f'{parts[0].strip()}\t{parts[1].strip()}\n')

    print(f'Cleaned: {filename}')

print('Cleaning done')

## 4. Language Tags
Prefix each source sentence with the language tag `>>lang<<` (e.g. `>>fra<<`).

In [ ]:
for filename in os.listdir(CLEAN_DIR):
    if not filename.endswith('.txt'):
        continue

    lang = os.path.splitext(filename)[0]
    tag  = f'>>{lang}<< '
    path = os.path.join(CLEAN_DIR, filename)

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        line = line.rstrip('\n')
        if not line.strip():
            continue
        parts = line.split('\t')
        if len(parts) != 2:
            continue
        src, tgt = parts
        new_lines.append(f'{tag}{src.strip()}\t{tgt.strip()}\n')

    with open(path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)

    print(f'Tag added: {filename}')

print('Tagging done')

## 5. CSV Construction
For each language file found in CLEAN_DIR, generate both directions:
- `>>fra<< [english]` → `[french]`  (same for spa, ara, por)
- `>>eng<< [language]` → `[english]`

All pairs are shuffled and split into train/valid/test (90/5/5).

In [ ]:
import random, csv, re

random.seed(42)

# remove punctuation from source sentence so the model learns to generate it in the target language
def normalize_src(sentence):
    sentence = re.sub(r"[?.!,;:،؟؛]", " ", sentence)
    return re.sub(r" +", " ", sentence).strip()

# isolate punctuation in target sentence so the model learns to place it correctly
def normalize_tgt(sentence):
    sentence = re.sub(r"([?.!,;:'،؟؛])", r" \1 ", sentence)
    return re.sub(r" +", " ", sentence).strip()

# filter out pairs that are too short or too long (noise + memory)
def ok_pair(src, tgt):
    src_toks = src.strip().split()
    tgt_toks = tgt.strip().split()
    if len(src_toks) < 2 or len(tgt_toks) < 1: return False
    if len(src_toks) > 40 or len(tgt_toks) > 40: return False
    return True

pairs = []

# loop over all language files found in CLEAN_DIR
for filename in os.listdir(CLEAN_DIR):
    if not filename.endswith('.txt'):
        continue

    # extract language code from filename (e.g. 'fra' from 'fra.txt')
    lang      = os.path.splitext(filename)[0]
    lang_path = os.path.join(CLEAN_DIR, filename)
    lang_pairs = 0

    with open(lang_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) != 2:
                continue
            raw_src, raw_tgt = parts

            # strip the existing tag (>>lang<<) to get the raw English text
            en_text   = re.sub(r'^>>\w+<<\s*', '', raw_src).strip()
            lang_text = raw_tgt.strip()

            # direction 1: English → Target language
            # the >>lang<< tag tells the model which language to produce
            en_src   = f'>>{lang}<< ' + normalize_src(en_text)
            lang_tgt = normalize_tgt(lang_text)
            if ok_pair(en_src, lang_tgt):
                pairs.append((en_src, lang_tgt))
                lang_pairs += 1

            # direction 2: Target language → English
            # the >>eng<< tag tells the model to produce English
            lang_src = '>>eng<< ' + normalize_src(lang_text)
            en_tgt   = normalize_tgt(en_text)
            if ok_pair(lang_src, en_tgt):
                pairs.append((lang_src, en_tgt))
                lang_pairs += 1

    print(f'{filename} → {lang_pairs} pairs (both directions)')

print(f'\nTotal: {len(pairs)} pairs across all languages')

# shuffle globally so every epoch sees all languages mixed together
random.shuffle(pairs)
n = len(pairs)
n_train = int(0.90 * n)
n_valid = int(0.05 * n)

def write_csv(name, data):
    out = os.path.join(DATA_DIR, f'{name}.csv')
    with open(out, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['source', 'target'])
        writer.writerows(data)
    print(f'Written: {out} ({len(data)} rows)')

write_csv('train', pairs[:n_train])
write_csv('valid', pairs[n_train:n_train + n_valid])
write_csv('test',  pairs[n_train + n_valid:])
print(f'\nSplit: {n_train} train | {n_valid} valid | {n - n_train - n_valid} test')

## 6. Model Training
**Seq2Seq LSTM** — Encoder / Decoder with Bahdanau Attention and Teacher Forcing.

In [ ]:
import csv, math, random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from google.colab import files

# hyperparameters
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LEN = 60 
MIN_FREQ = 3  
BATCH = 512 
EMB = 256 
HID = 512
NUM_LAYERS = 2  
DROPOUT = 0.3  
EPOCHS = 15
LR = 1e-3
TEACHER_FORCING = 0.5  
DOWNLOAD_AT = {5, 10, 15}
PAD, SOS, EOS, UNK = '<pad>', '<s>', '</s>', '<unk>'

random.seed(SEED)
torch.manual_seed(SEED)
print('DEVICE:', DEVICE)

# tokenizer : character-level for CJK scripts, word-level (split on spaces) for the rest
def tokenize(s):
    s = s.strip()
    if any('\u3000' <= c <= '\u9fff' or '\uac00' <= c <= '\ud7af' for c in s):
        return list(s.replace(' ', ''))
    return s.split()

def read_csv_pairs(path):
    pairs = []
    with open(path, 'r', encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            src = (row.get('source') or '').strip()
            tgt = (row.get('target') or '').strip()
            if src and tgt:
                pairs.append((src, tgt))
    return pairs

# Vocabulary: maps tokens <-> integer indices
class Vocab:
    def __init__(self, texts, min_freq=1):
        counter = Counter()
        for t in texts:
            counter.update(tokenize(t))
        self.itos = [PAD, SOS, EOS, UNK]  # special tokens always at index 0-3
        for w, c in counter.items():
            if c >= min_freq and w not in self.itos:
                self.itos.append(w)
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad = self.stoi[PAD]
        self.sos = self.stoi[SOS]
        self.eos = self.stoi[EOS]
        self.unk = self.stoi[UNK]

    def encode(self, s, add_sos=False, add_eos=False):
        ids = []
        if add_sos: ids.append(self.sos)
        for tok in tokenize(s)[:MAX_LEN]:
            ids.append(self.stoi.get(tok, self.unk))
        if add_eos: ids.append(self.eos)
        return ids

    def __len__(self):
        return len(self.itos)

class PairsDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.sv = src_vocab
        self.tv = tgt_vocab

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return (
            torch.tensor(self.sv.encode(src, add_eos=True), dtype=torch.long),
            torch.tensor(self.tv.encode(tgt, add_sos=True, add_eos=True), dtype=torch.long)
        )

# pad all sentences in a batch to the same length
def collate_fn(batch, src_pad, tgt_pad):
    srcs, tgts = zip(*batch)
    src_lens = torch.tensor([len(x) for x in srcs], dtype=torch.long)
    tgt_lens = torch.tensor([len(x) for x in tgts], dtype=torch.long)
    src_batch = torch.full((len(batch), src_lens.max()), src_pad, dtype=torch.long)
    tgt_batch = torch.full((len(batch), tgt_lens.max()), tgt_pad, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_batch[i, :len(s)] = s
        tgt_batch[i, :len(t)] = t
    return src_batch, src_lens, tgt_batch, tgt_lens

# Bahdanau Attention
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v    = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        B, S, _ = encoder_outputs.shape
        h_top  = hidden[-1].unsqueeze(1).repeat(1, S, 1)
        energy = torch.tanh(self.attn(torch.cat([h_top, encoder_outputs], dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1) 

# Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_id, num_layers, dropout):
        super().__init__()
        self.hid_dim    = hid_dim
        self.num_layers = num_layers
        self.emb        = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        # Bidirectional LSTM : reads the sentence left → right AND right → left simultaneously
        self.rnn        = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers,
                                  bidirectional=True, dropout=dropout, batch_first=True)
        self.fc_h       = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c       = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, src):
        B = src.size(0)
        embedded        = self.dropout(self.emb(src))
        outputs, (h, c) = self.rnn(embedded)
        # h : (num_layers*2, B, hid_dim) — separate layers and directions
        h = h.view(self.num_layers, 2, B, self.hid_dim)
        c = c.view(self.num_layers, 2, B, self.hid_dim)
        # Concatenate forward + backward, then project → (num_layers, B, hid_dim)
        h = torch.tanh(self.fc_h(torch.cat([h[:, 0], h[:, 1]], dim=2)))
        c = torch.tanh(self.fc_c(torch.cat([c[:, 0], c[:, 1]], dim=2)))
        return outputs, h, c

# Decoder
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, pad_id, num_layers, dropout):
        super().__init__()
        self.emb     = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attn    = Attention(enc_hid_dim, dec_hid_dim)
        self.rnn     = nn.LSTM(emb_dim + enc_hid_dim * 2, dec_hid_dim,
                               num_layers=num_layers, dropout=dropout, batch_first=True)
        self.fc      = nn.Linear(dec_hid_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inp, h, c, encoder_outputs):
        embedded     = self.dropout(self.emb(inp)).unsqueeze(1)
        attn_weights = self.attn(h, encoder_outputs).unsqueeze(1)
        context      = torch.bmm(attn_weights, encoder_outputs)  # weighted sum of encoder outputs
        rnn_input    = torch.cat([embedded, context], dim=2)
        out, (h, c)  = self.rnn(rnn_input, (h, c))
        return self.fc(out.squeeze(1)), h, c

# Seq2Seq 
class Seq2Seq(nn.Module):
    def __init__(self, enc, dec, device):
        super().__init__()
        self.enc    = enc
        self.dec    = dec
        self.device = device

    def forward(self, src, tgt, teacher_forcing=0.5):
        B, T = tgt.size()
        V    = self.dec.fc.out_features
        outputs           = torch.zeros(B, T, V, device=self.device)
        enc_outputs, h, c = self.enc(src)
        inp = tgt[:, 0]  # first input token is always <SOS>
        for t in range(1, T):
            logits, h, c  = self.dec(inp, h, c, enc_outputs)
            outputs[:, t] = logits
            top1 = logits.argmax(1)
            # teacher forcing : feed the correct word (train fast) or the predicted word (be autonomous)
            inp = tgt[:, t] if random.random() < teacher_forcing else top1
        return outputs

# training loop
def run_epoch(model, loader, optim, crit, train=True, ep=0):
    model.train(train)
    total_loss   = 0.0
    correct_toks = 0
    total_toks   = 0
    label = 'train' if train else 'valid'
    bar   = tqdm(loader, desc=f'epoch {ep} [{label}]', unit='batch')

    for src, src_lens, tgt, tgt_lens in bar:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        if train: optim.zero_grad()

        out    = model(src, tgt, teacher_forcing=TEACHER_FORCING if train else 0.0)
        logits = out[:, 1:].reshape(-1, out.size(-1))
        gold   = tgt[:, 1:].reshape(-1)
        loss   = crit(logits, gold) 

        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        mask          = gold != tgt_vocab.pad
        preds         = logits.argmax(1)
        correct_toks += (preds[mask] == gold[mask]).sum().item()
        total_toks   += mask.sum().item()

        total_loss += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / max(1, len(loader))
    accuracy = correct_toks / max(1, total_toks) * 100
    return avg_loss, accuracy

def save_checkpoint(path, model, src_vocab, tgt_vocab):
    torch.save({
        'model':      model.state_dict(),
        'src_itos':   src_vocab.itos,
        'tgt_itos':   tgt_vocab.itos,
        'emb':        EMB,
        'hid':        HID,
        'num_layers': NUM_LAYERS,
    }, path)

# main code
train_pairs = read_csv_pairs(os.path.join(DATA_DIR, 'train.csv'))
valid_pairs = read_csv_pairs(os.path.join(DATA_DIR, 'valid.csv'))

src_vocab = Vocab([s for s, _ in train_pairs], min_freq=MIN_FREQ)
tgt_vocab = Vocab([t for _, t in train_pairs], min_freq=MIN_FREQ)
print(f'Source vocab: {len(src_vocab)} | Target vocab: {len(tgt_vocab)}')

train_ds = PairsDataset(train_pairs, src_vocab, tgt_vocab)
valid_ds = PairsDataset(valid_pairs, src_vocab, tgt_vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad, tgt_vocab.pad))
valid_loader = DataLoader(valid_ds, batch_size=BATCH, shuffle=False,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad, tgt_vocab.pad))

enc   = Encoder(len(src_vocab), EMB, HID, src_vocab.pad, NUM_LAYERS, DROPOUT)
dec   = Decoder(len(tgt_vocab), EMB, HID, HID, tgt_vocab.pad, NUM_LAYERS, DROPOUT)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)
crit  = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad)
# reduce LR by half if valid loss doesn't improve for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, patience=3, factor=0.5)

history = {'train_loss': [], 'valid_loss': [], 'train_acc': [], 'valid_acc': []}

best = float('inf')
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, optim, crit, train=True,  ep=ep)
    va_loss, va_acc = run_epoch(model, valid_loader, optim, crit, train=False, ep=ep)
    ppl = math.exp(min(va_loss, 10))  # perplexity: lower = better

    scheduler.step(va_loss)

    history['train_loss'].append(tr_loss)
    history['valid_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['valid_acc'].append(va_acc)

    print(f'epoch {ep:>2} | train loss {tr_loss:.4f} | valid loss {va_loss:.4f} | ppl {ppl:.2f} | train acc {tr_acc:.1f}% | valid acc {va_acc:.1f}%')

    # save best model whenever valid loss improves
    if va_loss < best:
        best = va_loss
        save_checkpoint(CKPT_PATH, model, src_vocab, tgt_vocab)
        print(f'  Best model saved → {CKPT_PATH}')

    # download snapshots at specific epochs
    if ep in DOWNLOAD_AT:
        ep_path = f'/content/drive/MyDrive/checkpoint_ep{ep}.pt'
        save_checkpoint(ep_path, model, src_vocab, tgt_vocab)
        print(f'  Checkpoint epoch {ep} saved → {ep_path}')
        files.download(ep_path)
        print(f'  Download triggered: checkpoint_ep{ep}.pt')

# visualisation
epochs_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['valid_loss'], 'r-o', label='Valid Loss')
ax1.set_title('Loss per epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Accuracy')
ax2.plot(epochs_range, history['valid_acc'], 'r-o', label='Valid Accuracy')
ax2.set_title('Accuracy per epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png', dpi=150)
plt.show()
print('Graph saved to Drive!')

## 7. Inference — Translation
Load the best model checkpoint and translate a sentence.

In [ ]:
import os, re, torch, torch.nn as nn

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LEN = 60
PAD, SOS, EOS, UNK = '<pad>', '<s>', '</s>', '<unk>'

# load from local path if available, otherwise fall back to Drive
LOCAL_CKPT = os.path.join(os.getcwd(), 'models', 'lstm_seq2seq', 'best_multi.pt')
DRIVE_CKPT = '/content/drive/MyDrive/best_multi.pt'
CKPT_PATH  = LOCAL_CKPT if os.path.exists(LOCAL_CKPT) else DRIVE_CKPT
print('Loading from:', CKPT_PATH)

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v    = nn.Linear(dec_hid_dim, 1, bias=False)
    def forward(self, hidden, encoder_outputs):
        B, S, _ = encoder_outputs.shape
        h_top  = hidden[-1].unsqueeze(1).repeat(1, S, 1)
        energy = torch.tanh(self.attn(torch.cat([h_top, encoder_outputs], dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, pad_id, num_layers):
        super().__init__()
        self.hid_dim    = hid_dim
        self.num_layers = num_layers
        self.emb        = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.rnn        = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers,
                                  bidirectional=True, batch_first=True)
        self.fc_h       = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_c       = nn.Linear(hid_dim * 2, hid_dim)
    def forward(self, src):
        B = src.size(0)
        outputs, (h, c) = self.rnn(self.emb(src))
        h = h.view(self.num_layers, 2, B, self.hid_dim)
        c = c.view(self.num_layers, 2, B, self.hid_dim)
        h = torch.tanh(self.fc_h(torch.cat([h[:, 0], h[:, 1]], dim=2)))
        c = torch.tanh(self.fc_c(torch.cat([c[:, 0], c[:, 1]], dim=2)))
        return outputs, h, c

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, pad_id, num_layers):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attn = Attention(enc_hid_dim, dec_hid_dim)
        self.rnn  = nn.LSTM(emb_dim + enc_hid_dim * 2, dec_hid_dim,
                            num_layers=num_layers, batch_first=True)
        self.fc   = nn.Linear(dec_hid_dim, vocab_size)
    def forward(self, inp, h, c, encoder_outputs):
        embedded     = self.emb(inp).unsqueeze(1)
        attn_weights = self.attn(h, encoder_outputs).unsqueeze(1)
        context      = torch.bmm(attn_weights, encoder_outputs)
        rnn_input    = torch.cat([embedded, context], dim=2)
        out, (h, c)  = self.rnn(rnn_input, (h, c))
        return self.fc(out.squeeze(1)), h, c

# all supported translation directions, tag = target language code
DIRECTIONS = {
    '1': {'label': 'English → French',     'tag': 'fra', 'src_lang': 'English',    'tgt_lang': 'French',
          'placeholder': 'Ex: I love Paris.'},
    '2': {'label': 'French → English',     'tag': 'eng', 'src_lang': 'French',     'tgt_lang': 'English',
          'placeholder': "Ex: J'aime Paris."},
    '3': {'label': 'English → Spanish',    'tag': 'spa', 'src_lang': 'English',    'tgt_lang': 'Spanish',
          'placeholder': 'Ex: I love Madrid.'},
    '4': {'label': 'Spanish → English',    'tag': 'eng', 'src_lang': 'Spanish',    'tgt_lang': 'English',
          'placeholder': 'Ex: Amo Madrid.'},
    '5': {'label': 'English → Arabic',     'tag': 'ara', 'src_lang': 'English',    'tgt_lang': 'Arabic',
          'placeholder': 'Ex: I love you.'},
    '6': {'label': 'Arabic → English',     'tag': 'eng', 'src_lang': 'Arabic',     'tgt_lang': 'English',
          'placeholder': 'Ex: أنا أحبك.'},
    '7': {'label': 'English → Portuguese', 'tag': 'por', 'src_lang': 'English',    'tgt_lang': 'Portuguese',
          'placeholder': 'Ex: I love Lisbon.'},
    '8': {'label': 'Portuguese → English', 'tag': 'eng', 'src_lang': 'Portuguese', 'tgt_lang': 'English',
          'placeholder': 'Ex: Eu amo Lisboa.'},
}

# tokenizer : character-level for CJK, word-level for everything else
def tokenize(s):
    s = s.strip()
    if any('\u3000' <= c <= '\u9fff' or '\uac00' <= c <= '\ud7af' for c in s):
        return list(s.replace(' ', ''))
    return s.split()

def load_model(ckpt_path):
    ck         = torch.load(ckpt_path, map_location=DEVICE)
    src_itos   = ck['src_itos']
    tgt_itos   = ck['tgt_itos']
    src_stoi   = {w: i for i, w in enumerate(src_itos)}
    tgt_stoi   = {w: i for i, w in enumerate(tgt_itos)}
    emb        = ck['emb']
    hid        = ck['hid']
    num_layers = ck.get('num_layers', 1)
    enc = Encoder(len(src_itos), emb, hid, src_stoi[PAD], num_layers)
    dec = Decoder(len(tgt_itos), emb, hid, hid, tgt_stoi[PAD], num_layers)
    full = ck['model']
    enc.load_state_dict({k[4:]: v for k, v in full.items() if k.startswith('enc.')})
    dec.load_state_dict({k[4:]: v for k, v in full.items() if k.startswith('dec.')})
    enc.to(DEVICE).eval()
    dec.to(DEVICE).eval()
    return enc, dec, src_stoi, tgt_itos, tgt_stoi

# clean the source sentence before encoding
def preprocess(sentence):
    sentence = re.sub(r"[?.!,;:،؟؛]", " ", sentence)
    return re.sub(r" +", " ", sentence).strip()

# clean and reassemble the decoded token list into a readable string
def postprocess(tokens):
    tokens = [t for t in tokens if t != UNK]
    text = ' '.join(tokens)
    text = re.sub(r" ' ", "'", text)
    text = re.sub(r" ([.!?,;:])", r"\1", text)
    return text if text else '(model not trained enough yet)'

def translate(sentence, tag, enc, dec, src_stoi, tgt_itos, tgt_stoi):
    clean        = preprocess(sentence)
    # prepend the target language tag so the model knows what to produce
    src_with_tag = f'>>{tag}<< {clean}'
    ids = [src_stoi.get(t, src_stoi[UNK]) for t in tokenize(src_with_tag)]
    ids.append(src_stoi[EOS])
    src_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        enc_outputs, h, c = enc(src_tensor)
        inp = torch.tensor([tgt_stoi[SOS]], dtype=torch.long).to(DEVICE)
        result = []
        for _ in range(MAX_LEN):
            logits, h, c = dec(inp, h, c, enc_outputs)
            top1 = logits.argmax(1).item()  # greedy decoding : pick the most likely token
            if tgt_itos[top1] == EOS: break
            result.append(tgt_itos[top1])
            inp = torch.tensor([top1], dtype=torch.long).to(DEVICE)
    return postprocess(result)

enc, dec, src_stoi, tgt_itos, tgt_stoi = load_model(CKPT_PATH)
print('Model loaded!\n')
for k, d in DIRECTIONS.items():
    print(f'  {k}. {d["label"]}')

In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

dir_dropdown = widgets.Dropdown(
    options=[(d['label'], k) for k, d in DIRECTIONS.items()],
    description='Direction :',
    layout=widgets.Layout(width='350px')
)

sentence_input = widgets.Text(
    placeholder=DIRECTIONS['1']['placeholder'],
    description='Phrase :',
    layout=widgets.Layout(width='500px')
)

btn    = widgets.Button(description='Traduire', button_style='primary')
output = widgets.Output()

def on_direction_change(change):
    d = DIRECTIONS[change['new']]
    sentence_input.placeholder = d['placeholder']
    sentence_input.value = ''
    with output:
        clear_output()

dir_dropdown.observe(on_direction_change, names='value')

def on_click(b):
    with output:
        clear_output()
        sentence = sentence_input.value.strip()
        if not sentence:
            print("Écris une phrase !")
            return
        key       = dir_dropdown.value
        d         = DIRECTIONS[key]
        tag       = d['tag']
        src_label = d['src_lang']
        tgt_label = d['tgt_lang']
        translation = translate(sentence, tag, enc, dec, src_stoi, tgt_itos, tgt_stoi)
        print(f"{src_label} : {sentence}")
        print(f"{tgt_label} : {translation}")

btn.on_click(on_click)
display(dir_dropdown, sentence_input, btn, output)

Dropdown(description='Direction :', layout=Layout(width='350px'), options=(('Anglais → Français', '1'), ('Fran…

Text(value='', description='Phrase :', layout=Layout(width='500px'), placeholder='Ex: I love Paris.')

Button(button_style='primary', description='Traduire', style=ButtonStyle())

Output()